# SML Assignment 3
**Name**: 2024011


## Question 1: PCA, Ridge & Lasso Regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
import urllib.request
import os

def load_and_preprocess_mnist(p=10):
    path = "mnist.npz"
    if not os.path.exists(path):
        urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", path)
    with np.load(path) as data:
        train_images, train_labels = data['x_train'], data['y_train']
        test_images, test_labels = data['x_test'], data['y_test']
    train_mask = np.isin(train_labels, [0, 1, 2])
    test_mask = np.isin(test_labels, [0, 1, 2])
    X_train, y_train = train_images[train_mask], train_labels[train_mask]
    X_test, y_test = test_images[test_mask], test_labels[test_mask]
    X_train = X_train.reshape(X_train.shape[0], -1).astype(float) / 255.0
    X_test = X_test.reshape(X_test.shape[0], -1).astype(float) / 255.0
    mu = np.mean(X_train, axis=0, keepdims=True)
    Xc_train = X_train - mu
    Xc_test = X_test - mu
    S = (Xc_train.T @ Xc_train) / (Xc_train.shape[0] - 1)
    evals, evecs = np.linalg.eigh(S)
    idx = np.argsort(evals)[::-1]
    evecs = evecs[:, idx]
    W_pca = evecs[:, :p]
    X_train_pca = Xc_train @ W_pca
    X_test_pca = Xc_test @ W_pca
    return X_train_pca, y_train, X_test_pca, y_test, X_train, X_test, mu, evecs

X_train, y_train, X_test, y_test, X_train_raw, X_test_raw, mu_raw, evecs_raw = load_and_preprocess_mnist(p=10)


In [ ]:
X_train_aug = np.hstack([np.ones((X_train.shape[0], 1)), X_train])
X_test_aug = np.hstack([np.ones((X_test.shape[0], 1)), X_test])

lambdas = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]

def ridge_fit(X, y, lam):
    D = X.shape[1]
    I_mod = np.eye(D)
    I_mod[0, 0] = 0.0
    return np.linalg.inv(X.T @ X + lam * I_mod) @ X.T @ y

def get_targets(y, k):
    return (y == k).astype(float)

ridge_train_mse, ridge_test_mse = [], []
lasso_train_mse, lasso_test_mse = [], []
ridge_paths_class1, lasso_paths_class1, lasso_nonzeros = [], [], []

for lam in lambdas:
    r_tr_err = 0; r_te_err = 0
    l_tr_err = 0; l_te_err = 0
    ridge_weights_c1 = None
    lasso_weights_c1 = None
    nonzero_count = 0
    
    for k in [0, 1, 2]:
        yk_train = get_targets(y_train, k)
        yk_test = get_targets(y_test, k)
        
        W_r = ridge_fit(X_train_aug, yk_train, lam)
        r_tr_err += np.mean((X_train_aug @ W_r - yk_train)**2)
        r_te_err += np.mean((X_test_aug @ W_r - yk_test)**2)
        if k == 1: ridge_weights_c1 = W_r[1:]
            
        lasso = Lasso(alpha=lam, fit_intercept=True, max_iter=10000, tol=1e-4)
        lasso.fit(X_train, yk_train)
        l_tr_err += np.mean((lasso.predict(X_train) - yk_train)**2)
        l_te_err += np.mean((lasso.predict(X_test) - yk_test)**2)
        nonzero_count += np.sum(lasso.coef_ != 0)
        if k == 1: lasso_weights_c1 = lasso.coef_
            
    ridge_train_mse.append(r_tr_err / 3)
    ridge_test_mse.append(r_te_err / 3)
    ridge_paths_class1.append(ridge_weights_c1)
    
    lasso_train_mse.append(l_tr_err / 3)
    lasso_test_mse.append(l_te_err / 3)
    lasso_paths_class1.append(lasso_weights_c1)
    lasso_nonzeros.append(nonzero_count / 3)

ridge_paths_class1 = np.array(ridge_paths_class1)
lasso_paths_class1 = np.array(lasso_paths_class1)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(lambdas, ridge_train_mse, marker='o', label='Train MSE')
ax[0].plot(lambdas, ridge_test_mse, marker='s', label='Test MSE')
ax[0].set_xscale('log')
ax[0].set_title('Ridge Regression MSE (p=10)')
ax[0].legend()
ax[1].plot(lambdas, lasso_train_mse, marker='o', label='Train MSE')
ax[1].plot(lambdas, lasso_test_mse, marker='s', label='Test MSE')
ax[1].set_xscale('log')
ax[1].set_title('Lasso Regression MSE (p=10)')
ax[1].legend()
plt.show()

## Question 2: Decision Trees

In [ ]:
def get_gini(y):
    if len(y) == 0: return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1.0 - np.sum(p**2)

class DecisionNode:
    def __init__(self, data_indices):
        self.data_indices = data_indices
        self.feature_idx, self.threshold = None, None
        self.left, self.right = None, None
        self.is_leaf = True
        self.label = None

class CustomTree:
    def __init__(self, max_splits=2, max_features=None):
        self.max_splits = max_splits
        self.max_features = max_features
        self.root = None
        self.leaves = []

    def fit(self, X, y):
        self.root = DecisionNode(np.arange(len(y)))
        self.leaves = [self.root]
        overall_n = len(y)
        
        for _ in range(self.max_splits):
            best_split_info, best_overall_gini = None, float('inf')
            for leaf in self.leaves:
                leaf_X, leaf_y = X[leaf.data_indices], y[leaf.data_indices]
                if len(np.unique(leaf_y)) <= 1: continue
                features = np.arange(X.shape[1]) if self.max_features is None else np.random.choice(X.shape[1], self.max_features, replace=False)
                for f_idx in features:
                    threshold = np.mean(leaf_X[:, f_idx])
                    left_mask = leaf_X[:, f_idx] <= threshold
                    if np.sum(left_mask) == 0 or np.sum(~left_mask) == 0: continue
                    gini_split = (np.sum(left_mask)/overall_n)*get_gini(leaf_y[left_mask]) + (np.sum(~left_mask)/overall_n)*get_gini(leaf_y[~left_mask])
                    current_overall = sum([(len(other.data_indices)/overall_n)*get_gini(y[other.data_indices]) for other in self.leaves if other != leaf])
                    if current_overall + gini_split < best_overall_gini:
                        best_overall_gini = current_overall + gini_split
                        best_split_info = (leaf, f_idx, threshold, leaf.data_indices[left_mask], leaf.data_indices[~left_mask])
            if best_split_info:
                split_leaf, f_idx, thresh, left_idx, right_idx = best_split_info
                split_leaf.is_leaf = False
                split_leaf.feature_idx, split_leaf.threshold = f_idx, thresh
                split_leaf.left, split_leaf.right = DecisionNode(left_idx), DecisionNode(right_idx)
                self.leaves.remove(split_leaf)
                self.leaves.extend([split_leaf.left, split_leaf.right])
            else: break
                
        for leaf in self.leaves:
            vals, counts = np.unique(y[leaf.data_indices], return_counts=True)
            leaf.label = vals[np.argmax(counts)]
            
    def _predict_single(self, x, node):
        if node.is_leaf: return node.label
        return self._predict_single(x, node.left) if x[node.feature_idx] <= node.threshold else self._predict_single(x, node.right)

    def predict(self, X): return np.array([self._predict_single(x, self.root) for x in X])

tree = CustomTree(max_splits=2)
tree.fit(X_train, y_train)
print(f"Single Tree Accuracy: {np.mean(tree.predict(X_test) == y_test)*100:.2f}%")


## Question 3: Fashion-MNIST Stumps

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
(t_img, t_lbl), (te_img, te_lbl) = fashion_mnist.load_data()
tr_m, te_m = np.isin(t_lbl, [0, 1, 2]), np.isin(te_lbl, [0, 1, 2])
X_f_train, y_f_train = (t_img[tr_m].reshape(-1, 784)/255.0), t_lbl[tr_m].astype(float)
X_f_test, y_f_test = (te_img[te_m].reshape(-1, 784)/255.0), te_lbl[te_m].astype(float)

mu_f = np.mean(X_f_train, axis=0)
X_f_train_c = X_f_train - mu_f
S_f = (X_f_train_c.T @ X_f_train_c) / (len(X_f_train_c) - 1)
evals_f, evecs_f = np.linalg.eigh(S_f)
W_f = evecs_f[:, np.argsort(evals_f)[::-1]][:, :10]
X_f_train_p, X_f_test_p = X_f_train_c @ W_f, (X_f_test - mu_f) @ W_f

class RegStump:
    def fit(self, X, y):
        n_s, n_f = X.shape
        best_ssr = float('inf')
        for f_idx in range(n_f):
            sort_idx = np.argsort(X[:, f_idx])
            X_s, y_s = X[sort_idx, f_idx], y[sort_idx]
            sum_l, sq_l, sum_r, sq_r = 0.0, 0.0, np.sum(y_s), np.sum(y_s**2)
            for i in range(1, n_s):
                v = y_s[i-1]
                sum_l += v; sq_l += v**2; sum_r -= v; sq_r -= v**2
                if X_s[i-1] < X_s[i]:
                    ssr_t = (sq_l - sum_l**2/i) + (sq_r - sum_r**2/(n_s-i))
                    if ssr_t < best_ssr:
                        best_ssr = ssr_t
                        self.feature_idx, self.threshold = f_idx, (X_s[i-1] + X_s[i])/2
                        self.l_val, self.r_val = sum_l/i, sum_r/(n_s-i)
    def predict(self, X):
        p = np.zeros(X.shape[0])
        m = X[:, self.feature_idx] <= self.threshold
        p[m], p[~m] = self.l_val, self.r_val
        return p
stump = RegStump()
stump.fit(X_f_train_p, y_f_train)
print(f"Stump Test MSE: {np.mean((stump.predict(X_f_test_p) - y_f_test)**2):.4f}")
